In [3]:
import xarray as xr
import h5py
import pandas as pd
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
from os.path import join
import os
from pathlib import Path
from dateutil.parser import parse

from multiprocessing import Pool
from credit.pbs import get_num_cpus

Process MPAS H of x from `mpas_dir` and save it to `save_dir`

In [ ]:
# mpas_dir= "/glade/derecho/scratch/bjung/pandac/ForecastFromGFSAnalyses/Verification/fc/mean/2022120212/"
# mpas_dir= "/glade/derecho/scratch/ivette/pandac/v3.0.3/mpasjedi_goes_hofx_v3.0.3/Verification/fc/mean/2025061300"

mpas_dirs = list(Path("/glade/derecho/scratch/ivette/pandac/v3.0.3/mpasjedi_goes_hofx_v3.0.3/Verification/fc/mean/").iterdir())
print(f"converting dirs {mpas_dirs}")
parent_save_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/mpas_forecasts/"

def process_mpas_forecast(mpas_dir):
    save_dir = join(parent_save_dir, Path(mpas_dir).name)
    os.makedirs(save_dir, exist_ok=True)

    # reference coords
    zarr_ds = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr", consolidated=False)
    ref_lat = zarr_ds.latitude.sortby("latitude")
    ref_lon = zarr_ds.longitude.sortby("longitude")
    dirs = [p for p in Path(mpas_dir).iterdir() if p.is_dir() and "hr" in p.name]
    def save_mpas_output(file, save_dir):
        with h5py.File(file, 'r') as f:
            channels = f["Channel"][:]
            BT = f["hofx/brightnessTemperature"][:]
            latitude = f['MetaData/latitude'][:]
            longitude = f['MetaData/longitude'][:]
            time = pd.Timestamp(f["MetaData/dateTime"][0], unit="s").round('30min') - pd.Timedelta(5, "m")

        lat_unique = np.unique(latitude)
        lon_unique = np.unique(longitude)
        assert(len(lat_unique) == 1003)
        assert(len(lon_unique) == 923)
        n_lat = len(lat_unique)
        n_lon = len(lon_unique)
        
        # Create empty 2D grid
        data_2d = np.full((n_lat, n_lon, len(channels)), np.nan)
        
        # Fill the grid by matching coordinates
        for (lat_val, lon_val, data_val) in zip(latitude, longitude, BT):
            # Find indices in the sorted arrays
            lat_idx = np.searchsorted(lat_unique, lat_val)
            lon_idx = np.searchsorted(lon_unique, lon_val)
            
            # Place data in correct position
            data_2d[lat_idx, lon_idx] = data_val
        
        da = xr.DataArray(
            data=data_2d,
            coords={
                "latitude": ref_lat,
                "longitude": ref_lon,
                "channel": channels,
            },
            dims = ["latitude", "longitude", "channel"],
        ).expand_dims({"t": [time]}).transpose("t", "channel", ...)
        
        ds = da.to_dataset(name="BT_or_R")
        
        time_str = time.strftime("%Y-%m-%dT%H:%M:%S")
        ds.to_netcdf(join(save_dir, f"{time_str}.nc"))

    for directory in dirs: 
        try:
            save_mpas_output(join(directory, "dbOut/obsout_hofx_abi_g19.h5"), save_dir)
        except:
            print(f"error in {directory}")
            pass
        
    # print(f"finished saving to {save_dir}")


converting dirs ['/glade/derecho/scratch/ivette/pandac/v3.0.3/mpasjedi_goes_hofx_v3.0.3/Verification/fc/mean/2025070318']


In [ ]:
num_cpus = get_num_cpus()
print(f"using {num_cpus} cpus")
with Pool(num_cpus - 1) as p:
    p.map(process_mpas_forecast, mpas_dirs)
    p.close()
    p.join()

finished saving to /glade/derecho/scratch/dkimpara/goes_10km_train/mpas_forecasts/2025070318


#file for testing purposes
test_file= "/glade/derecho/scratch/bjung/pandac/ForecastFromGFSAnalyses/Verification/fc/mean/2022120212/0hr30mn/dbOut/obsout_hofx_abi_g16.h5"

with h5py.File(test_file, 'r') as f:
    # List all top-level groups and datasets in the file
    for key in f.keys():
        obj = f[key]
        if isinstance(obj, h5py.Dataset):
            print(f"{key}: {obj.shape} {obj.dtype}")
        else:
            print(f"{key}/: Group")

    print(f["hofx/brightnessTemperature"].shape)
    group = f['MetaData']
    print("Group members:", list(group.keys()))
    print(group["latitude"][:])
    print(group["longitude"])

In [6]:
parent_save_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/mpas_forecasts/"

dirs = list(Path(parent_save_dir).iterdir())
for folder_path in dirs:
    folder = folder_path.name
    year, month, day, hour = int(folder[:4]), int(folder[4:6]), int(folder[6:8]), int(folder[8:10])
    t_init = pd.Timestamp(year=year, month=month, day=day, hour=hour)
    t_str = t_init.strftime("%Y-%m-%dT%H:%M:%S")
    os.rename(folder_path, join(parent_save_dir, t_str))